# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadSulaiman08/FLYRANK-ML-INTERNSHIP/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**One row = one page (content_hash_id) on one day (report_date)** in `fact_content_daily_performance`, for the mid-panel month **March 2026** (`month=2026-03`). I do not use the `_sample` table, because it is the final month and must stay a sealed test month.

**Windows:** features come from **Mar 1–15** and the label from **Mar 16–31**, so the decision moment is the end of Mar 15. **Label (proxy):** a page "declined" if its daily impressions in Mar 16–31 fell more than 20% below its Mar 1–15 daily average. I only keep pages with ≥ 50 impressions in Mar 1–15, so tiny pages don't make noise.## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## 2. Fields: feature / label / context / excluded

- **Features (5, all from Mar 1–15):** impressions, clicks, CTR, average position, days seen.
- **Label source:** `gsc_impressions` in Mar 16–31, used only to build the label, never as a feature.
- **Context:** `client_hash_id` (grouping and client-held-out splits), `content_hash_id` (the unit), `report_date` (defines the windows), `ga4_data_available` (availability flag).
- **Excluded:** (1) anything from Mar 16–31, because it's the future at the decision moment (leakage); (2) GA4 session columns for now, because they are zero-filled before each client's GA4 start, so a zero there doesn't mean "no traffic". I stay GSC-only in this first contract.## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [39]:
contract = pd.DataFrame([
    ("client_hash_id",      "context",  "grouping + client-held-out splits only, never a feature"),
    ("content_hash_id",     "context",  "the unit: one page"),
    ("report_date",         "context",  "defines the windows (Mar 1-15 = features, Mar 16-31 = label)"),
    ("gsc_impressions",     "feature + label source", "Mar 1-15 sums -> features; Mar 16-31 -> label only"),
    ("gsc_clicks",          "feature",  "Mar 1-15 only"),
    ("gsc_avg_position",    "feature",  "Mar 1-15 only"),
    ("ga4_data_available",  "context",  "availability flag; GA4 zeros before a client's GA4 start are fill, not 'no traffic'"),
    ("GA4 session columns", "excluded", "missing/zero-filled for clients before GA4 start -> I stay GSC-only for now"),
    ("any Mar 16-31 value", "excluded", "that is the label window = future at the decision moment (leakage)"),
], columns=["field", "bucket", "why"])
have = set(cols["column_name"])
contract["in_table"] = contract["field"].map(lambda f: "yes" if f in have else "-")
contract

,field,bucket,why,in_table
0,client_hash_id,context,"grouping + client-held-out splits only, never ...",yes
1,content_hash_id,context,the unit: one page,yes
2,report_date,context,"defines the windows (Mar 1-15 = features, Mar ...",yes
3,gsc_impressions,feature + label source,Mar 1-15 sums -> features; Mar 16-31 -> label ...,yes
4,gsc_clicks,feature,Mar 1-15 only,yes
5,gsc_avg_position,feature,Mar 1-15 only,yes
6,ga4_data_available,context,availability flag; GA4 zeros before a client's...,yes
7,GA4 session columns,excluded,missing/zero-filled for clients before GA4 sta...,-
8,any Mar 16-31 value,excluded,that is the label window = future at the decis...,-


## 3. Verify it with queries (grain, counts, missing values, windows)

Three verification queries on March 2026, then my five-feature frame and the deliberate leak.

- **Grain:** [0] duplicate (date, client, content) rows, so one row really is one page-day.
- **Count + span:** [N] rows, [N] clients, [N] pages, from [first day] to [last day].
- **Availability:** [N] of [N] rows ([X]%) survive `ga4_data_available IS TRUE`. That confirms GA4 isn't available everywhere, which is why I stay GSC-only.
- **Five features**, each knowable at the end of Mar 15 (one line each in the output).
- **The trap:** adding `imp_16_31` (impressions from the label window) pushed ROC AUC from [honest] to [leaky]. It was "too good" because the label is computed from that column. I deleted it and kept the honest number, [honest].

In [40]:
q1 = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM {MONTH}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print("Query 1 - GRAIN: duplicate (date, client, content) rows found:", len(q1))
print("-> 0 means one row really is one page on one day." if len(q1) == 0 else q1)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 1 - GRAIN: duplicate (date, client, content) rows found: 0
-> 0 means one row really is one page on one day.


In [41]:
q2 = con.sql(f"""
    SELECT COUNT(*) AS rows, COUNT(DISTINCT client_hash_id) AS clients,
           COUNT(DISTINCT content_hash_id) AS pages,
           MIN(report_date) AS first_day, MAX(report_date) AS last_day
    FROM {MONTH}
""").df()
print("Query 2 - ROW COUNT + DATE SPAN of my slice (March 2026):")
q2

Query 2 - ROW COUNT + DATE SPAN of my slice (March 2026):


,rows,clients,pages,first_day,last_day
0,9841378,55,331437,2026-03-01,2026-03-31


In [42]:
q3 = con.sql(f"""
    SELECT COUNT(*) AS all_rows,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS rows_ga4_available,
           ROUND(100.0 * COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) / COUNT(*), 1) AS pct_survive,
           COUNT(*) FILTER (WHERE gsc_impressions > 0) AS rows_with_impressions
    FROM {MONTH}
""").df()
print("Query 3 - AVAILABILITY: rows that survive `ga4_data_available IS TRUE`")
q3

Query 3 - AVAILABILITY: rows that survive `ga4_data_available IS TRUE`


,all_rows,rows_ga4_available,pct_survive,rows_with_impressions
0,9841378,413966,4.2,3611061


In [43]:
frame = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
        SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_1_15,
        SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_clicks      ELSE 0 END) AS clk_1_15,
        AVG(CASE WHEN report_date <= DATE '2026-03-15' AND gsc_avg_position > 0 THEN gsc_avg_position END) AS pos_1_15,
        COUNT(DISTINCT CASE WHEN report_date <= DATE '2026-03-15' AND gsc_impressions > 0 THEN report_date END) AS days_seen_1_15,
        SUM(CASE WHEN report_date >  DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_16_31
    FROM {MONTH}
    GROUP BY 1, 2
    HAVING imp_1_15 >= 50
""").df()

frame["ctr_1_15"] = frame["clk_1_15"] / frame["imp_1_15"]
# Label: daily impressions in Mar 16-31 fell more than 20% vs Mar 1-15
frame["declined"] = ((frame["imp_16_31"] / 16) < 0.8 * (frame["imp_1_15"] / 15)).astype(int)

FEATURES = {
    "imp_1_15":       "knowable on Mar 15: impressions already recorded Mar 1-15",
    "clk_1_15":       "knowable on Mar 15: clicks already recorded Mar 1-15",
    "ctr_1_15":       "knowable on Mar 15: computed only from Mar 1-15 clicks/impressions",
    "pos_1_15":       "knowable on Mar 15: average position over Mar 1-15 (0 = no data, excluded)",
    "days_seen_1_15": "knowable on Mar 15: how many of the first 15 days the page appeared",
}
frame["pos_1_15"] = frame["pos_1_15"].fillna(frame["pos_1_15"].median())
print(f"{len(frame):,} pages with >= 50 impressions in Mar 1-15 | declined share: {frame['declined'].mean():.3f}\n")
for f, why in FEATURES.items():
    print(f"{f:15} <- {why}")
frame[["content_hash_id"] + list(FEATURES) + ["declined"]].head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

92,548 pages with >= 50 impressions in Mar 1-15 | declined share: 0.327

imp_1_15        <- knowable on Mar 15: impressions already recorded Mar 1-15
clk_1_15        <- knowable on Mar 15: clicks already recorded Mar 1-15
ctr_1_15        <- knowable on Mar 15: computed only from Mar 1-15 clicks/impressions
pos_1_15        <- knowable on Mar 15: average position over Mar 1-15 (0 = no data, excluded)
days_seen_1_15  <- knowable on Mar 15: how many of the first 15 days the page appeared


,content_hash_id,imp_1_15,clk_1_15,ctr_1_15,pos_1_15,days_seen_1_15,declined
0,content_39d7361b4945d504,57.0,0.0,0.000000,4.222711,15,1
1,content_cec711b02f3bbde6,199.0,2.0,0.010050,4.086084,13,0
2,content_275b6f7f733016d4,467.0,1.0,0.002141,4.449176,13,1
3,content_ceaec531566ffcfc,56.0,0.0,0.000000,7.700694,14,1
4,content_755d951187fcd70a,771.0,1.0,0.001297,1.883472,14,0


In [44]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

y, g = frame["declined"].values, frame["client_hash_id"].values
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42).split(frame, y, g))

def quick_auc(cols):
    m = DecisionTreeClassifier(max_depth=4, min_samples_leaf=50, random_state=42)
    m.fit(frame.iloc[tr][cols], y[tr])
    return roc_auc_score(y[te], m.predict_proba(frame.iloc[te][cols])[:, 1])

honest = list(FEATURES)
leaky = honest + ["imp_16_31"]          # TRAP: impressions from the label window itself
print(f"Honest 5 features      ROC AUC (unseen clients): {quick_auc(honest):.3f}")
print(f"+ imp_16_31 (LEAK!)    ROC AUC (unseen clients): {quick_auc(leaky):.3f}   <- 'too good': the label is built from this column")
# Remove the leak: it is never used again
frame = frame.drop(columns=["imp_16_31"])
print("\nLeak removed. Honest number kept:", round(quick_auc(honest), 3))

Honest 5 features      ROC AUC (unseen clients): 0.602
+ imp_16_31 (LEAK!)    ROC AUC (unseen clients): 0.788   <- 'too good': the label is built from this column

Leak removed. Honest number kept: 0.602


## 4. Data limits

**Named limitation: unbalanced history.** Clients start at different dates: only [N] of [N] clients have 12+ months of search history (query below). So one global window treats a client that started tracking recently as if it had "no traffic" before. My March-only slice also can't see seasonality, and a half-month vs half-month comparison is noisy, since a holiday or a single spike can trigger "declined". GSC-only means I can't say anything about engagement yet.

In [45]:
lim = con.sql(f"""
    SELECT COUNT(*) AS clients,
           COUNT(*) FILTER (WHERE gsc_data_start <= DATE '2025-03-01') AS clients_with_12m_plus_history,
           COUNT(*) FILTER (WHERE gsc_data_start IS NULL)               AS clients_no_gsc_start
    FROM {CLIENTS}
""").df()
print("Limitation check - history depth differs per client:")
lim

Limitation check - history depth differs per client:


,clients,clients_with_12m_plus_history,clients_no_gsc_start
0,104,3,37


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.